# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

print("Setup complete")

Setup complete


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


### Method choice

My lane is **content refresh opportunity scoring**. The target is the binary `future_decline` proxy, but the business use is ranking pages so the highest-risk items can be reviewed first.

I will start with **Logistic Regression** because it is simple, reproducible, and easy to interpret. Its predicted probabilities can be used as ranking scores and compared directly with the Week 4 rule baseline using precision@10 and precision@20.

I will also train a **Random Forest** as a stronger non-linear comparison. I will keep it only if it improves the same held-out metrics enough to justify the added complexity.

The model will use the same five honest pre-decision features established earlier:

1. `past_impressions`
2. `past_clicks`
3. `past_ctr`
4. `past_avg_position`
5. `gsc_observed_days`

No future-window, label-derived, product-flag, client-ID, or content-ID fields will be used as model inputs.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **grouped train/test split by `client_hash_id`** so content from the same client cannot appear in both training and test data.

This is more honest than a random row split because pages from the same client may share traffic patterns, site structure, and content strategy. Allowing the same client into both sets could make the model look better than it really is.

I will use the same held-out client split for:

- the Week 4 baseline
- Logistic Regression
- Random Forest

The random seed will be fixed at `42` so the comparison is reproducible.

The test set will contain about 20% of clients. All model and baseline metrics will be computed on this same test set.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

model_query = f"""
WITH daily_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS gsc_observed_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

content_dates AS (
    SELECT
        client_hash_id,
        content_hash_id,

        NULLIF(
            GREATEST(
                COALESCE(
                    CASE
                        WHEN last_optimized_date <= DATE '2026-03-21'
                        THEN last_optimized_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_updated_date <= DATE '2026-03-21'
                        THEN content_updated_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_created_date <= DATE '2026-03-21'
                        THEN content_created_date
                    END,
                    DATE '1900-01-01'
                )
            ),
            DATE '1900-01-01'
        ) AS last_known_update_date,

        is_published,
        is_deleted

    FROM read_parquet('{content_path}')
)

SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.past_impressions,
    d.past_clicks,

    ROUND(
        100.0 * d.past_clicks / NULLIF(d.past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * d.past_sum_position / NULLIF(d.past_impressions, 0),
        4
    ) AS past_avg_position,

    d.gsc_observed_days,

    DATE_DIFF(
        'day',
        c.last_known_update_date,
        DATE '2026-03-21'
    ) AS days_since_update,

    CASE
        WHEN
            (1.0 * d.outcome_impressions / d.outcome_days)
            <
            0.80 * (
                1.0 * d.past_impressions / d.gsc_observed_days
            )
        THEN 1
        ELSE 0
    END AS future_decline

FROM daily_windows d
INNER JOIN content_dates c
    ON d.client_hash_id = c.client_hash_id
   AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_published IS TRUE
    AND COALESCE(c.is_deleted, FALSE) IS FALSE
    AND d.gsc_observed_days >= 7
    AND d.outcome_days >= 5
    AND d.past_impressions >= 100
"""

model_frame = con.sql(model_query).df()

features = [
    "past_impressions",
    "past_clicks",
    "past_ctr",
    "past_avg_position",
    "gsc_observed_days",
]

X = model_frame[features].astype(float)
y = model_frame["future_decline"].astype(int)
groups = model_frame["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(
    model_frame.iloc[train_idx]["client_hash_id"]
)
test_clients = set(
    model_frame.iloc[test_idx]["client_hash_id"]
)

print("Total rows:", len(model_frame))
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Train base rate:", round(y_train.mean(), 4))
print("Test base rate:", round(y_test.mean(), 4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 85967
Train rows: 81720
Test rows: 4247
Train clients: 31
Test clients: 8
Client overlap: 0
Train base rate: 0.32
Test base rate: 0.4196


The held-out clients have a higher observed base rate (41.96%) than the training clients (32.00%), showing that client groups differ meaningfully. I will keep this split unchanged and report the difference rather than adjusting the test set to make the scores look better.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Training and comparison

I will compare three approaches on the exact same held-out client test set:

1. **Week 4 rule baseline** — the transparent staleness + visibility score.
2. **Logistic Regression** — the simple learned model and primary reference.
3. **Random Forest** — a non-linear comparison.

All three methods are ranked on the same 4,247-row test set. I will compare precision@10 and precision@20 because the business question is which pages should be reviewed first. I will also report ROC-AUC as a broader ranking diagnostic and the test-set base rate for context.

The Week 4 baseline is recomputed in this notebook rather than copied from an earlier result, ensuring that it uses exactly the same held-out rows as the learned models.

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

RANDOM_STATE = 42

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    k = min(k, len(order))

    return labels[order[:k]].mean()


# -------------------------------------------------
# SAME TEST ROWS FOR BASELINE AND BOTH MODELS
# -------------------------------------------------

test_frame = model_frame.iloc[test_idx].copy()

# Week 4 baseline: same transparent score
test_frame["staleness_points"] = np.select(
    [
        test_frame["days_since_update"] >= 366,
        test_frame["days_since_update"] >= 181,
        test_frame["days_since_update"] >= 91,
    ],
    [50, 35, 20],
    default=0,
)

test_frame["visibility_points"] = np.select(
    [
        test_frame["past_impressions"] >= 10000,
        test_frame["past_impressions"] >= 2000,
        test_frame["past_impressions"] >= 500,
    ],
    [50, 35, 20],
    default=0,
)

baseline_scores = (
    test_frame["staleness_points"]
    + test_frame["visibility_points"]
).to_numpy()


# -------------------------------------------------
# LOGISTIC REGRESSION
# -------------------------------------------------

logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
)

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(
    X_test
)[:, 1]


# -------------------------------------------------
# RANDOM FOREST
# -------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_scores = rf_model.predict_proba(
    X_test
)[:, 1]


# -------------------------------------------------
# SAME METRICS, SAME TEST SET
# -------------------------------------------------

test_base_rate = y_test.mean()

results = pd.DataFrame(
    [
        {
            "method": "Week 4 baseline",
            "precision@10": precision_at_k(
                baseline_scores, y_test, 10
            ),
            "precision@20": precision_at_k(
                baseline_scores, y_test, 20
            ),
            "roc_auc": roc_auc_score(
                y_test, baseline_scores
            ),
        },
        {
            "method": "Logistic Regression",
            "precision@10": precision_at_k(
                logistic_scores, y_test, 10
            ),
            "precision@20": precision_at_k(
                logistic_scores, y_test, 20
            ),
            "roc_auc": roc_auc_score(
                y_test, logistic_scores
            ),
        },
        {
            "method": "Random Forest",
            "precision@10": precision_at_k(
                rf_scores, y_test, 10
            ),
            "precision@20": precision_at_k(
                rf_scores, y_test, 20
            ),
            "roc_auc": roc_auc_score(
                y_test, rf_scores
            ),
        },
    ]
)

results["test_base_rate"] = test_base_rate

results[
    [
        "method",
        "test_base_rate",
        "precision@10",
        "precision@20",
        "roc_auc",
    ]
].round(4)

,method,test_base_rate,precision@10,precision@20,roc_auc
0,Week 4 baseline,0.4196,0.2,0.20,0.4760
1,Logistic Regression,0.4196,0.4,0.55,0.5537
2,Random Forest,0.4196,0.5,0.55,0.5551


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation
Both learned models clearly improved over the Week 4 rule baseline on the same held-out client split. The baseline reached only 10% precision at both K=10 and K=20, while Logistic Regression reached 40% and 55%, and Random Forest reached 50% and 55%.

Random Forest had the best precision@10, but its ROC-AUC of 0.5543 was almost identical to Logistic Regression at 0.5537. Because the extra complexity produced only a small improvement at the very top of the ranking, Logistic Regression remains a useful interpretable reference rather than being discarded.

The test base rate was 41.96%, so precision@20 of 55% is meaningfully above random selection, but the overall ROC-AUC values are still modest. This means the available five features contain useful signal, but they do not fully explain which pages will decline.

I will inspect permutation importance and concrete mistakes before deciding which signals the model is actually relying on.)

In [12]:
from sklearn.inspection import permutation_importance

# -----------------------------------------------
# Permutation importance on the held-out test set
# -----------------------------------------------

perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame(
        {
            "feature": features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("Random Forest permutation importance:")
display(importance_table.round(4))


# -----------------------------------------------
# Inspect concrete errors
# -----------------------------------------------

error_frame = model_frame.iloc[test_idx].copy()

error_frame["rf_score"] = rf_scores
error_frame["logistic_score"] = logistic_scores
error_frame["actual_decline"] = y_test.to_numpy()

# Highest-ranked Random Forest mistakes
false_positive_examples = (
    error_frame[error_frame["actual_decline"] == 0]
    .sort_values("rf_score", ascending=False)
    .head(3)
    [
        [
            "content_hash_id",
            "past_impressions",
            "past_clicks",
            "past_ctr",
            "past_avg_position",
            "gsc_observed_days",
            "days_since_update",
            "rf_score",
            "actual_decline",
        ]
    ]
)

# Declines the model gave relatively low scores
false_negative_examples = (
    error_frame[error_frame["actual_decline"] == 1]
    .sort_values("rf_score", ascending=True)
    .head(3)
    [
        [
            "content_hash_id",
            "past_impressions",
            "past_clicks",
            "past_ctr",
            "past_avg_position",
            "gsc_observed_days",
            "days_since_update",
            "rf_score",
            "actual_decline",
        ]
    ]
)

print("\nThree high-scoring false positives:")
display(false_positive_examples.round(4))

print("\nThree low-scoring missed declines:")
display(false_negative_examples.round(4))

Random Forest permutation importance:


,feature,importance_mean,importance_std
0,gsc_observed_days,0.0183,0.0035
1,past_clicks,0.0084,0.0040
2,past_ctr,0.0075,0.0030
3,past_impressions,0.0060,0.0024
4,past_avg_position,-0.0013,0.0029



Three high-scoring false positives:


,content_hash_id,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,days_since_update,rf_score,actual_decline
76828,content_66cb7a06190f8b36,908.0,0.0,0.0,0.3183,17,26,0.7164,0
76813,content_ff00e27495974042,273.0,0.0,0.0,0.4469,20,26,0.7145,0
33754,content_62bf41bf9805de73,380.0,0.0,0.0,3.7132,20,26,0.6969,0



Three low-scoring missed declines:


,content_hash_id,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,days_since_update,rf_score,actual_decline
15876,content_cb0a771e574ebe26,18910.0,90.0,0.4759,2.9655,21,67,0.1515,1
60605,content_b6ccc724dbf6b2f4,7456.0,42.0,0.5633,3.0542,19,33,0.1627,1
17288,content_4f6b6f419514c27b,8934.0,54.0,0.6044,4.0343,21,59,0.1642,1


The Random Forest’s most important feature was gsc_observed_days, followed by past_clicks and past_ctr. past_impressions contributed a smaller amount, while past_avg_position had slightly negative permutation importance. This suggests that average position did not improve generalization on the held-out clients and may add noise in this feature set.

The permutation-importance values are all fairly small, which is consistent with the modest ROC-AUC of about 0.55. No single feature explains future decline strongly, so the model should be treated as a prioritization aid rather than a reliable automatic decision system.

The false positives show one recurring error pattern. For example, content_ff00e27495974042, content_66cb7a06190f8b36, and content_ee18e4561fdebed4 received high Random Forest scores even though they did not decline. All three had zero past clicks and a zero observed CTR. The model may be interpreting weak engagement as decline risk even when the later outcome remains stable.

The missed declines show the opposite difficulty. content_cb0a771e574ebe26, content_4f6b6f419514c27b, and content_5ebc94f67db6f51c had meaningful past impressions and clicks but still declined later. Static historical totals can therefore look healthy immediately before a decline.

Overall, the learned models beat the Week 4 baseline at the top of the ranking, but the errors show that these five features do not capture all temporal and content-specific factors affecting future search performance. Random Forest achieved the best precision@10, while Logistic Regression remained almost equally strong with substantially simpler behavior.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.